In [ ]:
import pandas as pd

## **Cargamos datos "procesados"**

In [ ]:
df_jan = pd.read_parquet("data/processed/jan.parquet")
df_feb = pd.read_parquet("data/processed/feb.parquet")


## **EDA**

In [ ]:
import matplotlib.pyplot as plt

# Distribución de la variable objetivo
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_jan["duration"], bins=50)
ax.set_title("Distribución de duration (train)")
ax.set_xlabel("duration (min)")
ax.set_ylabel("count")
fig.tight_layout()
plt.show()

# Missingness simple
missing = df_jan.isna().mean().sort_values(ascending=False).head(10)
missing

## **Definir columnas a modelar**

In [ ]:
categorical = ['PULocationID', 'DOLocationID']
numerical = ["trip_distance"]

## Definir set de train y test

In [ ]:
df_train = df_jan
df_val = df_feb

## **Preprocesar**

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

df_train = df_train.copy()
df_val   = df_val.copy()
df_train["categorical_dict"] = df_train[categorical].to_dict(orient="records")
df_val["categorical_dict"]   = df_val[categorical].to_dict(orient="records")

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", DictVectorizer(), "categorical_dict"),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), numerical),
    ]
)

X_train = preprocessor.fit_transform(df_train)
y_train = df_train["duration"].values
X_val   = preprocessor.transform(df_val)
y_val   = df_val["duration"].values









## **Entrenar**

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=0, max_depth=10)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)


## **Metricas de modelado**

In [ ]:
from sklearn.metrics import root_mean_squared_error


rmse = root_mean_squared_error(y_val, y_pred)

In [ ]:
rmse

## **Next Steps**

## Ejercicio 1 (equipos - sala de Zoom)

**Modalidad:** equipos de 2-3 personas, sala de breakout en Zoom.
**Tiempo sugerido:** 10 minutos.

Acaban de entrenar un modelo sin ningun tipo de registro. En equipo, respondan y anoten
(alguien debe compartir pantalla y escribir las respuestas):

1. Si el resultado de `rmse` de esta celda fuera el mejor que han obtenido hasta ahora,
   ?como se lo demostrarian a alguien mas manana, si ya cerraron esta notebook?
2. Si quisieran comparar este RandomForest contra un XGBoost y contra 5 combinaciones
   distintas de hiperparametros, ?que tendrian que hacer manualmente hoy para no perder
   el rastro de cada intento?
3. Nombren al menos 3 piezas de informacion que NO quedaron guardadas en ningun lado
   (pista: parametros, version de datos, artefactos, codigo exacto).

**Entregable para la plenaria:** un representante del equipo comparte 1 de las 3
respuestas del punto 3 en voz alta.

En el siguiente notebook resolvemos exactamente este problema con **MLflow experiment
tracking**.
